# Interactive discrete wavelet transform viewer for sEEG (Google Colab)

This notebook loads a locally uploaded **`sEEG-HFOs-8.edf`**, removes every channel whose name starts with `MKR`, and provides an interactive DWT time-series viewer for every remaining channel.

Run the cells from top to bottom. Use the **Mother wavelet** control to change the wavelet used for decomposition.


## 1. Install dependencies

The quiet install is safe to run in Google Colab. After installation, Colab can read EDF files through MNE and calculate discrete wavelet transforms with PyWavelets.


In [ ]:
%pip install -q mne edfio PyWavelets ipywidgets


## 2. Imports and plotting defaults


In [ ]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import mne
import numpy as np
import pywt
from IPython.display import display

mne.set_log_level("WARNING")
plt.rcParams.update({"axes.grid": True, "figure.figsize": (15, 8)})


## 3. Locate or upload the EDF

The notebook first checks common Colab paths. If the file is not present, the browser upload dialog opens automatically. Select **`sEEG-HFOs-8.edf`**.


In [ ]:
EDF_NAME = "sEEG-HFOs-8.edf"
candidates = [
    Path("/content") / EDF_NAME,
    Path("/content/dataset") / EDF_NAME,
    Path.cwd() / EDF_NAME,
    Path.cwd() / "dataset" / EDF_NAME,
]
edf_path = next((path for path in candidates if path.is_file()), None)

if edf_path is None:
    from google.colab import files

    print(f"Select {EDF_NAME} in the upload dialog.")
    uploaded = files.upload()
    if EDF_NAME not in uploaded:
        raise FileNotFoundError(f"The uploaded file must be named {EDF_NAME!r}.")
    edf_path = Path("/content") / EDF_NAME

print(f"EDF file: {edf_path.resolve()}")


## 4. Load all signal channels and exclude MKR channels

The EDF remains disk-backed (`preload=False`), so moving the time slider reads only the requested segment. Filtering is case-insensitive and excludes names such as `MKR1+` and `MKR2+`.


In [ ]:
raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="WARNING")

marker_channels = [
    name for name in raw.ch_names
    if name.strip().upper().startswith("MKR")
]
signal_channels = [
    name for name in raw.ch_names
    if not name.strip().upper().startswith("MKR")
]
if not signal_channels:
    raise ValueError("No non-MKR signal channels were found in the EDF.")

raw.pick(signal_channels)
sfreq = float(raw.info["sfreq"])
duration_s = raw.n_times / sfreq

print(f"Signal channels available: {len(signal_channels)}")
print(f"Excluded MKR channels: {marker_channels or 'none found'}")
print(f"Sampling frequency: {sfreq:g} Hz")
print(f"Duration: {duration_s:.2f} seconds")
display(signal_channels)


## 5. Interactive DWT viewer

Choose any non-MKR channel, mother wavelet, decomposition level, start time, and window length. The first panel is the original signal. The remaining panels are time-aligned reconstructed approximation and detail components.

Approximate frequency limits are shown beside each component. These dyadic limits are descriptive; the actual response depends on the selected mother wavelet.


In [ ]:
def reconstructed_dwt_components(signal, mother_wavelet, requested_level):
    """Reconstruct approximation and details at the original sample alignment."""
    wavelet = pywt.Wavelet(mother_wavelet)
    maximum_level = pywt.dwt_max_level(len(signal), wavelet.dec_len)
    level = min(requested_level, maximum_level)
    if level < 1:
        raise ValueError("This time window is too short for the selected mother wavelet.")

    coefficients = pywt.wavedec(signal, wavelet, level=level, mode="symmetric")
    components = []
    for index in range(len(coefficients)):
        isolated = [np.zeros_like(coefficient) for coefficient in coefficients]
        isolated[index] = coefficients[index]
        reconstructed = pywt.waverec(isolated, wavelet, mode="symmetric")
        components.append(reconstructed[: len(signal)])

    labels = [f"A{level}"] + [f"D{detail}" for detail in range(level, 0, -1)]
    return labels, components, level


def component_frequency_label(label):
    """Return the approximate dyadic frequency range for a DWT component."""
    level = int(label[1:])
    if label.startswith("A"):
        return f"0–{sfreq / (2 ** (level + 1)):.1f} Hz"
    low = sfreq / (2 ** (level + 1))
    high = sfreq / (2 ** level)
    return f"{low:.1f}–{high:.1f} Hz"


channel_picker = widgets.Dropdown(
    options=signal_channels,
    value=signal_channels[0],
    description="Channel:",
    layout=widgets.Layout(width="320px"),
)

# Every discrete PyWavelets wavelet is available, not just a fixed shortlist.
mother_wavelet_picker = widgets.Dropdown(
    options=pywt.wavelist(kind="discrete"),
    value="db4",
    description="Mother wavelet:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
level_picker = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="DWT levels:", continuous_update=False,
    style={"description_width": "initial"},
)
window_picker = widgets.Dropdown(
    options=[1.0, 2.0, 5.0, 10.0, 20.0, 30.0],
    value=5.0,
    description="Window (s):",
    style={"description_width": "initial"},
)
start_picker = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=max(0.0, duration_s - window_picker.value),
    step=max(0.05, min(0.5, duration_s / 2000)),
    description="Start (s):",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)


def update_start_range(change=None):
    start_picker.max = max(0.0, duration_s - window_picker.value)
    start_picker.value = min(start_picker.value, start_picker.max)


window_picker.observe(update_start_range, names="value")


def plot_dwt(channel, mother_wavelet, levels, start, window):
    stop = min(start + window, duration_s)
    data, times = raw.get_data(
        picks=[channel], tmin=start, tmax=stop, return_times=True
    )
    signal_uv = data[0] * 1e6

    try:
        labels, components, used_level = reconstructed_dwt_components(
            signal_uv, mother_wavelet, levels
        )
    except ValueError as error:
        print(error)
        return

    figure, axes = plt.subplots(
        len(components) + 1,
        1,
        figsize=(15, max(7, 1.6 * (len(components) + 1))),
        sharex=True,
        constrained_layout=True,
    )
    axes[0].plot(times, signal_uv, color="black", linewidth=0.7)
    axes[0].set_ylabel("Original\n(µV)")
    axes[0].set_title(
        f"{channel} — DWT with {mother_wavelet} mother wavelet, "
        f"level {used_level} ({start:.2f}–{stop:.2f} s)"
    )

    for axis, label, component in zip(axes[1:], labels, components):
        axis.plot(times[: len(component)], component, linewidth=0.7)
        axis.set_ylabel(f"{label}\n{component_frequency_label(label)}\n(µV)")

    axes[-1].set_xlabel("Time (seconds)")
    for axis in axes:
        axis.margins(x=0)
    plt.show()


viewer = widgets.interactive_output(
    plot_dwt,
    {
        "channel": channel_picker,
        "mother_wavelet": mother_wavelet_picker,
        "levels": level_picker,
        "start": start_picker,
        "window": window_picker,
    },
)

display(
    widgets.HBox([channel_picker, mother_wavelet_picker]),
    widgets.HBox([level_picker, window_picker]),
    start_picker,
    viewer,
)
